# Preprocessing

## Package Imports

In [15]:
import numpy as np
import pandas as pd
from pickle import dump, load
import os

## Dataset Loading

In [16]:
with open('data/eda/data.pkl', 'rb') as file:
    df = load(file)

display(df.head())

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
0,747,15787619,Hsieh,844,France,Male,18,2,160980.03,145936.28,0
1,1620,15770309,McDonald,656,France,Male,18,10,151762.74,127014.32,0
2,1679,15569178,Kharlamov,570,France,Female,18,4,82767.42,71811.90,0
3,2022,15795519,Vasiliev,716,Germany,Female,18,3,128743.80,197322.13,0
4,2137,15621893,Bellucci,727,France,Male,18,4,133550.67,46941.41,0


## Defense Against Missing Predictor

In this context, we are going to exclude RowNumber, CustomerId, and Surname. However, input column might have more columns than our input here.

In [17]:
mp_check = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'Exited']
cols = list(df.columns)
not_exist = []
for i in mp_check:
    if i not in cols:
        not_exist.append(i)

if len(not_exist)>0:
    raise ValueError(f"Some columns are missing: {not_exist}")
else:    
    print("All required columns exist.")

All required columns exist.


## Handling Against Wrong Data Type

Here we also include RowNumber, CustomerId, and Surname

In [18]:
wt_check = {
    'RowNumber': 'int64',
    'CustomerId': 'int64',
    'Surname': 'object',
    'CreditScore': 'int64',
    'Geography': 'object',
    'Gender': 'object',
    'Age': 'int64',
    'Tenure': 'int64',
    'Balance': 'float64',
    'EstimatedSalary': 'float64',
    'Exited': 'int64'
}

for i in cols:
    if df[i].dtype != wt_check[i]:
        print(f'Column {i} has wrong data type')

Sometimes the issue comes from several entries that not match intended data types. Heres some approaches to deal with them.

* Numericals on categorical features would be set as NaN. We will deal with NaN categoricals on modelling phase.

In [19]:
def str_check(x):
    if isinstance(x, str):
        return x
    else:
        return np.nan
    
df['Surname'] = df['Surname'].apply(str_check)
df['Geography'] = df['Geography'].apply(str_check)
df['Gender'] = df['Gender'].apply(str_check)

* Strings on numerical features is handled by `pd.to_numeric`. Note that this would work with '56' but not something like 'Age: 56' or 'FiftySix'

In [20]:
df['RowNumber'] = pd.to_numeric(df['RowNumber'], errors='coerce').astype('int64')
df['CustomerId'] = pd.to_numeric(df['CustomerId'], errors='coerce').astype('int64')
df['CreditScore'] = pd.to_numeric(df['CreditScore'], errors='coerce').astype('int64')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce').astype('int64')
df['Tenure'] = pd.to_numeric(df['Tenure'], errors='coerce').astype('int64')
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')
df['EstimatedSalary'] = pd.to_numeric(df['EstimatedSalary'], errors='coerce')
df['Exited'] = pd.to_numeric(df['Exited'], errors='coerce').astype('int64')

In [21]:
## check if our data type match our requirement
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(6), object(3)
memory usage: 859.5+ KB
None


## Handling against missing data and outliers

For missing data:
* A data is considered to be missing if it is missing outside of RowNumber
* A row with missing data on CustomerId, Surname, and/or Exited wuld be removed
* Missing data on CreditScore, Age, Tenure, Balance, and EstimatedSalary would be ignored since `HistGradientBoostingClassifier` has native NaN support
* Entries not recognized on Geograpahy and Gender would be set as NaN, despite being well-written. This part would be explained in more detail during modelling phase

Note that we have no missing data at EDA phase, so no special treatment against that here.

For outlier data, here we simply remove it from our loaded data. Note that raw data can be accessed on data/eda

In [22]:
## load outlier data
with open('data/eda/out_data.pkl', 'rb') as file:
    out_df = load(file)

display(out_df.head())

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
1047,1194,15779947,Thomas,363,Spain,Female,28,6,146098.43,100615.14,1
1293,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,119346.88,1
1735,2580,15597896,Ozoemena,365,Germany,Male,30,0,127760.07,81537.85,1
4570,9211,15792650,Watts,382,Spain,Male,36,0,0.00,179540.73,1
5638,1839,15758813,Campbell,350,Germany,Male,39,0,109733.20,123602.11,1


In [23]:
df_diff = df.merge(out_df, how='left',
                   left_index=True, right_index=True,
                   indicator=True)

in_col = [i+'_y' for i in cols]
in_col.append('_merge')

in_df = df_diff[df_diff['_merge'] == 'left_only']
in_df = in_df.drop(in_col, axis=1)

in_col = [i+'_x' for i in cols]
in_dict = dict(zip(in_col, cols))

in_df = in_df.rename(in_dict, axis=1)
display(in_df)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
0,747,15787619,Hsieh,844,France,Male,18,2,160980.03,145936.28,0
1,1620,15770309,McDonald,656,France,Male,18,10,151762.74,127014.32,0
2,1679,15569178,Kharlamov,570,France,Female,18,4,82767.42,71811.90,0
3,2022,15795519,Vasiliev,716,Germany,Female,18,3,128743.80,197322.13,0
4,2137,15621893,Bellucci,727,France,Male,18,4,133550.67,46941.41,0
...,...,...,...,...,...,...,...,...,...,...,...
9636,9333,15638882,Cardell,710,Germany,Female,62,9,148214.36,48571.14,1
9637,9583,15742285,Andersen,559,France,Male,62,6,118756.62,20367.68,0
9638,9674,15784148,Beneventi,643,France,Male,62,9,0.00,155870.82,0
9639,9719,15704053,T'ang,710,Spain,Male,62,3,131078.42,119348.76,1


## Handling Against Imbalance and/or Skewed Data

* Imbalanced data are treated using stratify mechanism and `class_weight` parameter available for sklearn's `HistGradientBoostingClassifier`
* No additional treatment on skewed data since we are going to use Tree-based ensembles.

## Saving preprocessed data

In [24]:
if not os.path.exists('data/preprocessed'):
    os.makedirs('data/preprocessed')

with open('data/preprocessed/data.pkl', 'wb') as file:
    dump(in_df, file)